In [ ]:
def extract_text_from_context (context:dict):
    try:
        context_list = context["context_list"]
        joined_context = []
        for item in context_list:
            text = item.get("text", "")
            base64_string = item.get("base64_string")
            if base64_string:
                joined_context.append({
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_string}",
                        "detail": "high"
                    }
                })
            if text:
                joined_context.append({
                    "type": "text",
                    "text": text
                })
        context = joined_context
        #context = " ".join([item["text"] for item in context_list])
        #logger.info("text of the context is joined")
        logger.info(f"context joined is {context}")
        return context
    except Exception as e:
        logger.exception(f"Exception combining context {e}")
        raise HTTPException(status_code=500, detail=f"Internal server error: {e}") from e

: 

In [1]:
import pandas as pd

# Method 1: Read all tables and select by index
dfs = pd.read_html('can4_000.html')

print(f"Total tables found: {len(dfs)}")

# Preview all tables to see which one you need
for i, table in enumerate(dfs):
    print(f"\n--- Table {i} ---")
    print(f"Shape: {table.shape}")
    print(table.head(2))  # Show first 2 rows

# Select specific table by index (0-based)
df = dfs[0]  # First table
# df = dfs[1]  # Second table
# df = dfs[-1]  # Last table

print("\n=== Selected DataFrame ===")
print(df)
print("\nDataFrame Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

# ===== Method 2: Select table by matching content =====

# Select table containing specific text in headers
for i, table in enumerate(dfs):
    if 'ColumnName' in table.columns:  # Replace with your column name
        df = table
        print(f"Found table at index {i}")
        break

# Select table with specific dimensions
for i, table in enumerate(dfs):
    if table.shape[0] > 10:  # Tables with more than 10 rows
        df = table
        print(f"Found table with {table.shape[0]} rows at index {i}")
        break

# ===== Method 3: Using BeautifulSoup with attributes/class/id =====
from bs4 import BeautifulSoup

with open('your_file.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, 'html.parser')

# Find table by ID
table = soup.find('table', {'id': 'myTableId'})

# Find table by class
table = soup.find('table', {'class': 'myTableClass'})

# Find all tables and select by index
tables = soup.find_all('table')
table = tables[0]  # Select first table

# Find table containing specific text
for table in soup.find_all('table'):
    if 'specific text' in table.text:
        break

# Extract data from selected table
headers = [th.text.strip() for th in table.find_all('th')]
rows = []
for tr in table.find_all('tr')[1:]:
    row = [td.text.strip() for td in tr.find_all('td')]
    if row:
        rows.append(row)

df = pd.DataFrame(rows, columns=headers)
print("\nDataFrame from specific table:")
print(df)

# ===== Additional useful operations =====

# Save to CSV if needed
# df.to_csv('output.csv', index=False)

# Display basic statistics
# print("\nBasic Statistics:")
# print(df.describe())

# Check for missing values
# print("\nMissing Values:")
# print(df.isnull().sum())


IndexError: list index out of range

In [2]:
# Diagnostic: Check if HTML file contains tables
from bs4 import BeautifulSoup

with open('can4_000.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

print('First 500 characters of HTML:')
print(html_content[:500])

soup = BeautifulSoup(html_content, 'html.parser')
tables = soup.find_all('table')
print(f'Number of <table> tags found: {len(tables)}')

if tables:
    print('First table preview:')
    print(tables[0].prettify()[:500])
else:
    print('No tables found. Check the HTML structure.')

First 500 characters of HTML:
<!DOCTYPE html>
 <html>
  <head>
   <meta content="IE=Edge" http-equiv="X-UA-Compatible" charset="UTF-8"/>
   <title>can4_000</title>
   <script type="text/javascript" src="C:/Program Files/Tektronix/TekScope/CommonFiles/Stylesheets/jquery-1.8.2.min.js"></script>
   <script type="text/javascript" src="C:/Program Files/Tektronix/TekScope/CommonFiles/Stylesheets/easyzoom.js"></script>
   <script src="C:/Program Files/Tektronix/TekScope/CommonFiles/Stylesheets/ZoomImage1.js"></script>
   <script ty
Number of <table> tags found: 35
First table preview:
<table class="Table" id="Scope Details" name="Scope Details">
 <thead>
  <tr class="MeasName">
   <th colspan="4">
    Scope Details
   </th>
  </tr>
  <tr>
   <th class="ConfigHeader">
    Scope Model Number
   </th>
   <th class="ConfigHeader">
    Scope Serial Number
   </th>
   <th class="ConfigHeader">
    TekScope Version
   </th>
   <th class="ConfigHeader">
    Scope Calibration Status
   </th>
  </tr>
 

In [11]:
# Extract the first table using BeautifulSoup and convert to DataFrame
from bs4 import BeautifulSoup
import pandas as pd

with open('can4_000.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, 'html.parser')
tables = soup.find_all('table')

if tables:
    table = tables[4]  # Change index if you want a different table
    # Try to extract headers
    headers = [th.get_text(strip=True) for th in table.find_all('th')]
    rows = []
    for tr in table.find_all('tr')[1:]:
        cells = tr.find_all(['td', 'th'])
        row = [cell.get_text(strip=True) for cell in cells]
        if row:
            rows.append(row)
    # If headers are present and match row length, use them
    if headers and all(len(row) == len(headers) for row in rows):
        df = pd.DataFrame(rows, columns=headers)
    else:
        df = pd.DataFrame(rows)
    print('Extracted DataFrame:')
    pd.set_option('display.max_rows', None)
    print(df)
else:
    print('No tables found in the HTML.')

Extracted DataFrame:
      Name                Meas Sources        Mean'         Min'         Max'  \
0    Meas1           Amplitude    Ch 1     1.0577 V     1.0577 V     1.0577 V   
1    Meas2             Maximum    Ch 1     3.6927 V     3.6927 V     3.6927 V   
2    Meas3             Minimum    Ch 1     2.4553 V     2.4553 V     2.4553 V   
3    Meas4  Positive Overshoot    Ch 1     9.3677 %     9.3677 %     9.3677 %   
4    Meas5  Negative Overshoot    Ch 1     7.6112 %     7.6112 %     7.6112 %   
5    Meas6                 Top    Ch 1     3.5936 V     3.5936 V     3.5936 V   
6    Meas7                Base    Ch 1     2.5358 V     2.5358 V     2.5358 V   
7    Meas8           Data Rate    Ch 1  4.9989 Mb/s  4.7166 Mb/s  5.3183 Mb/s   
8    Meas9           Rise Time    Ch 1    39.816 ns    37.624 ns    41.514 ns   
9   Meas11           Fall Time    Ch 1    24.858 ns    24.283 ns    25.583 ns   
10  Meas12           Amplitude    Ch 2     1.1039 V     1.1039 V     1.1039 V   
11  Mea